# Signal Extraction — Solutions

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy  as np
import tables as tb
import pandas as pd
import matplotlib.pyplot as plt

import scipy.constants as constants
import scipy.stats     as stats
import scipy.optimize  as optimize

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, sys
from pathlib import Path

# Find the project root: honours FANAL_ROOT env-var, otherwise walks up from cwd
_env = os.environ.get('FANAL_ROOT') or os.environ.get('USCFANALDIR')
if _env and Path(_env, 'data').is_dir():
    rootpath = str(Path(_env).resolve())
else:
    rootpath = str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'data').is_dir() and (p / 'ana').is_dir()))
if rootpath not in sys.path:
    sys.path.insert(0, rootpath)
print('Fanal root : ', rootpath)

In [ ]:
import core.pltext  as pltext   # extensions for plotting histograms
import core.hfit    as hfit     # extension to fit histograms
import core.efit    as efit     # Fit Utilites - Includes Extend Likelihood Fit with composite PDFs
import core.utils   as ut       # generic utilities
import ana.fanal    as fn       # analysis functions specific to fanal
import ana.pltfanal as pltfn    # plotting for fanal
import     collpars as collpars # collaboration specific parameters
pltext.style()

In [ ]:
coll          = collpars.collaboration
sel_ntracks   = collpars.sel_ntracks
sel_eblob2    = collpars.sel_eblob2
sel_erange    = collpars.sel_erange
sel_eroi      = collpars.sel_eroi

print('Collaboration             : {:s}'.format(coll))
print('number of tracks range    : {:6d}'.format(sel_ntracks))
print('Blob-2 energy range       : {:6.3f}  MeV'.format(sel_eblob2))
print('Energy range              : ({:6.3f}, {:6.3f}) MeV'.format(*sel_erange))
print('Energy RoI range          : ({:6.3f}, {:6.3f}) MeV'.format(*sel_eroi))

In [ ]:
# list of the analisys selection variables names and ranges
ntracks_range = (sel_ntracks, sel_ntracks + 0.1)
eblob2_range  = (sel_eblob2, sel_erange[1]) # MeV

varnames  = ['num_tracks', 'blob2_E', 'E']
varranges = [ntracks_range, eblob2_range, sel_erange]
print('ana varnames  : ', varnames)
print('ana varranges : ', varranges)

# list of the reference selction variable names and rages to get pdfs for the MC
refnames  = ['num_tracks', 'E']
refranges = [ntracks_range, sel_erange]
print('ref varnames  : ', refnames)
print('ref varranges : ', refranges)


In [ ]:
# number of  blind events
n_Bi_total = collpars.n_Bi_total
n_Tl_total = collpars.n_Tl_total

eff_Bi_E   = collpars.eff_Bi_E
eff_Tl_E   = collpars.eff_Tl_E
eff_bb_E   = collpars.eff_bb_E
print('Number of bkg events in full data : Bi = {:6.2f}, Tl = {:6.2f}.'.format(n_Bi_total, n_Tl_total))

In [ ]:
# set the path to the data directory and filenames
dirpath = rootpath+'/data/'
filename = 'fanal_' + coll + '.h5'
print('Data path and filename : ', dirpath + filename)

# access the simulated data (DataFrames) for the different samples (Bi, Tl, bb) located in the data file
mcbi = pd.read_hdf(dirpath + filename, key = 'mc/bi214')
mctl = pd.read_hdf(dirpath + filename, key = 'mc/tl208')
mcbb = pd.read_hdf(dirpath + filename, key = 'mc/bb0nu')

# set the names of the samples
# set the names of the samples
mc_samples         = [mcbi, mctl, mcbb] # list of the mc DFs
sample_names       = ['Bi', 'Tl', 'bb']
sample_names_latex = [ r'$^{214}$Bi', r'$^{208}$Tl', r'$\beta\beta0\nu$',] # str names of the mc samples

for i, mc in enumerate(mc_samples):
    print('MC Sample {:s}, number of simulated events = {:d}'.format(sample_names[i], len(mc)))

## Generate experiments, fit signal, compute half-life

In [ ]:
def nevts_total(factor = 1.):
    """ returns the total number of events (Bi, Tl, bb), 
    The expected number of bb events in the RoI is a **factor = 1.** of the number of the expected Bi events in RoI
    """
    n_Bi_RoI    = collpars.n_Bi_RoI
    eff_bb_RoI  = collpars.eff_bb_RoI
    n_Bi_total  = collpars.n_Bi_total
    n_Tl_total  = collpars.n_Tl_total
    n_bb_total  = factor * n_Bi_RoI/eff_bb_RoI
    nevts       = (n_Bi_total, n_Tl_total, n_bb_total)
    return np.array(nevts)

def nevts_in_E(n_total):
    """ returns the number of expected events in the enlarged energy windows: (Bi, Tl, bb) 
    given a total number of events (Bi, Tl, bb)
    """
    eff_Bi_E      = collpars.eff_Bi_E
    eff_Tl_E      = collpars.eff_Tl_E
    eff_bb_E      = collpars.eff_bb_E
    nBi, nTl, nbb = n_total
    nevts         = (nBi * eff_Bi_E, nTl * eff_Tl_E, nbb * eff_bb_E)
    return np.array(nevts)

In [ ]:
# set the number of events
factor      = 0.5
n_total     = nevts_total(factor)
n_E         = nevts_in_E(n_total)

# define the fit
fit         = fn.prepare_fit_ell(mc_samples, n_total, varnames, varranges, refnames, refranges)

# generate an experiment using the mc samples and a given number of events of each sample
mcdata      = fn.generate_mc_experiment(mc_samples, n_total)

result, values, \
ell, n_est  = fit(mcdata)
n_est = result.x if result.success else n_E

pltfn.plot_fit_ell(values, n_est, ell, parnames = sample_names_latex)

In [ ]:
from collpars import exposure

eff      = 1. # total signal selection efficiency in the E window
n_bb_E   = n_est[-1] # estimated number of bb  in E
tau      = 1e25 ## compute the value of the lifetime

print('number of bb0nu events in E  : {:6.3f}'.format(n_bb_E))
print('exposure                     : {:6.2f} kg y'.format(exposure))
print('total signal efficiency      : {:6.3f}'.format(eff))
print('bb0nu half-life              : {:6.2e} y '.format(tau))
